<h1 align="left">Llama 3.2 Vision and TSViT Experiments</h1>

<p align="left">
  ITESM
  
  <a href="https://www.linkedin.com/in/juanrtato/">Juan Ricardo Albarracin B.</a>
  <br>
  <a href="">Luis Ángel Oporto Añacato.</a>
  <br>
  <a href="">David Alexis García Espinosa.</a>
  <br>
  <b>Last updated:</b> <i>30/05/2025</i>
  <br><br>
  <a target="_blank">
    <img src="https://github.com/QData/TextAttack/workflows/Github%20PyTest/badge.svg" alt="Testing">
  </a>
  <a href="https://img.shields.io/badge/version-0.1.0-blue.svg?cacheSeconds=2592000">
    <img src="https://img.shields.io/badge/version-0.1.0-blue.svg?cacheSeconds=2592000" alt="Version" height="18">
  </a>
</p><br>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import userdata
MyHF_TOKEN = userdata.get('HuggingFace_TECmx')

In [ ]:
import os
DIR = "/content/drive/MyDrive/Colab Notebooks/TecMNA/TC5035_PI/scripts"
os.chdir(DIR)

In [ ]:
import torch
from torch import nn
import numpy as np
from PIL import Image
import ast
import json
import random

CONFIG_PATH = '../datalake/config_vtt.json'
MODEL_PATH = '../datalake/TSViT/best.pth'

## Config Files

In [ ]:
from tsvit import torch_utils

In [ ]:
with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)

In [ ]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.6.0+cu124
12.4
True


In [ ]:
device_ids = [0]
device = torch_utils.get_device(device_ids, allow_cpu=False)

## PASTIS Data Loader

In [ ]:
from pastis24 import get_dataloaders

In [ ]:
with open("../datalake/label_names_en.json", "r") as json_file:
    label_names_en = json.load(json_file)
with open("../datalake/colormap.txt", "r") as txt_file:
    colormap = txt_file.readlines()
colormap = [ast.literal_eval(line.strip().rstrip(',')) for line in colormap]

In [ ]:
pastis_dataloader = get_dataloaders(config)

Loading PASTIS2SEQUENCE dataset...
PASTIS2SEQUENCE dataset loaded!


In [ ]:
sample = next(iter(pastis_dataloader['eval']))
sample_idx = random.randint(0, 16)
inputs = sample[0]['inputs']
labels = sample[0]['labels']

In [ ]:
print(inputs.shape)
print(inputs[sample_idx].shape)
inputs[sample_idx]

torch.Size([24, 60, 24, 24, 11])
torch.Size([60, 24, 24, 11])


tensor([[[[ 2.0560e+00,  1.8589e+00,  1.8250e+00,  ...,  8.8014e-01,
            8.3787e-01,  4.1096e-02],
          [ 1.9757e+00,  1.8079e+00,  1.8214e+00,  ...,  8.4450e-01,
            8.2171e-01,  4.1096e-02],
          [ 1.9088e+00,  1.7277e+00,  1.7346e+00,  ...,  7.8350e-01,
            7.8245e-01,  4.1096e-02],
          ...,
          [ 2.0040e+00,  1.8122e+00,  1.6984e+00,  ...,  6.1489e-01,
            5.4309e-01,  4.1096e-02],
          [ 1.9896e+00,  1.8010e+00,  1.7142e+00,  ...,  6.1351e-01,
            5.3078e-01,  4.1096e-02],
          [ 1.9876e+00,  1.8053e+00,  1.7270e+00,  ...,  6.0255e-01,
            5.1615e-01,  4.1096e-02]],

         [[ 2.1188e+00,  1.9248e+00,  1.9286e+00,  ...,  1.0015e+00,
            9.6255e-01,  4.1096e-02],
          [ 2.0282e+00,  1.8616e+00,  1.9107e+00,  ...,  9.6171e-01,
            9.5024e-01,  4.1096e-02],
          [ 1.9242e+00,  1.7521e+00,  1.7913e+00,  ...,  8.9179e-01,
            9.0868e-01,  4.1096e-02],
          ...,
     

## Unsloth Test

In [ ]:
# Do this only in Colab notebooks! Otherwise use pip install unsloth
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
!pip install transformers==4.51.3
!pip install --no-deps unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.7/145.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2025.5.

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Llama-3.2-11B-Vision-Instruct",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: We'll be using `/tmp/unsloth_compiled_cache` for temporary Unsloth patches.
Standard import failed for UnslothBCOTrainer: No module named 'UnslothBCOTrainer'. Using tempfile instead!
==((====))==  Unsloth 2025.5.8: Fast Mllama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/375k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.15k [00:00<?, ?B/s]

In [ ]:
model

MllamaForConditionalGeneration(
  (vision_model): MllamaVisionModel(
    (patch_embedding): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), padding=valid, bias=False)
    (gated_positional_embedding): MllamaPrecomputedPositionEmbedding(
      (tile_embedding): Embedding(9, 8197120)
    )
    (pre_tile_positional_embedding): MllamaPrecomputedAspectRatioEmbedding(
      (embedding): Embedding(9, 5120)
    )
    (post_tile_positional_embedding): MllamaPrecomputedAspectRatioEmbedding(
      (embedding): Embedding(9, 5120)
    )
    (layernorm_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (layernorm_post): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): MllamaVisionEncoder(
      (layers): ModuleList(
        (0-12): 13 x MllamaVisionEncoderLayer(
          (self_attn): MllamaVisionSdpaAttention(
            (q_proj): Linear4bit(in_features=1280, out_features=1280, bias=False)
            (k_proj): Linear4bit(in_features=1280, out_features

## HuggingFace Test

In [ ]:
!pip install "transformers>=4.45.0"
!pip install pillow

In [ ]:
!pip install huggingface_hub[hf_xet]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 44.1 MB/s eta 0:00:00


In [ ]:
import requests
from PIL import Image
from transformers import AutoProcessor, MllamaForConditionalGeneration
import torch

vision_model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
processor = AutoProcessor.from_pretrained(vision_model_id, token=MyHF_TOKEN)
model = MllamaForConditionalGeneration.from_pretrained(vision_model_id, torch_dtype=torch.bfloat16, token=MyHF_TOKEN)

config.json:   0%|          | 0.00/5.07k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:
model

MllamaForConditionalGeneration(
  (model): MllamaModel(
    (vision_model): MllamaVisionModel(
      (patch_embedding): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), padding=valid, bias=False)
      (gated_positional_embedding): MllamaPrecomputedPositionEmbedding(
        (tile_embedding): Embedding(9, 8197120)
      )
      (pre_tile_positional_embedding): MllamaPrecomputedAspectRatioEmbedding(
        (embedding): Embedding(9, 5120)
      )
      (post_tile_positional_embedding): MllamaPrecomputedAspectRatioEmbedding(
        (embedding): Embedding(9, 5120)
      )
      (layernorm_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (layernorm_post): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (transformer): MllamaVisionEncoder(
        (layers): ModuleList(
          (0-31): 32 x MllamaVisionEncoderLayer(
            (self_attn): MllamaVisionAttention(
              (q_proj): Linear(in_features=1280, out_features=1280, bias=False)
           

### Test Interacting with Llama-3.2-Vision

In [ ]:
# Example image URL
url = "https://miro.medium.com/v2/resize:fit:2400/1*nzdYUSs4c2RQs2W0FCHv1g.jpeg"
image = Image.open(requests.get(url, stream=True).raw)

# Prepare the image input with a text query
messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Can you describe this image?"}
    ]}
]

In [ ]:
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(image, input_text, add_special_tokens=False, return_tensors="pt").to(model.device)

In [ ]:
inputs.keys()

dict_keys(['input_ids', 'attention_mask', 'pixel_values', 'aspect_ratio_ids', 'aspect_ratio_mask', 'cross_attention_mask'])

In [ ]:
inputs['input_ids']

tensor([[128000, 128006,    882, 128007,    271, 128256,   6854,    499,   7664,
            420,   2217,     30, 128009, 128006,  78191, 128007,    271]])

In [ ]:
inputs['attention_mask']

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

In [ ]:
# batch_size, num_concurrent_media, num_tiles, num_channels, height, width = pixel_values.shape
print(inputs['pixel_values'].shape)
inputs['pixel_values']

torch.Size([1, 1, 4, 3, 560, 560])


tensor([[[[[[ 1.8135,  1.8135,  1.8135,  ...,  1.9011,  1.9011,  1.9011],
            [ 1.8135,  1.8135,  1.8135,  ...,  1.9011,  1.9011,  1.9011],
            [ 1.8135,  1.8135,  1.8135,  ...,  1.9011,  1.9011,  1.9011],
            ...,
            [ 1.7260,  1.7260,  1.7260,  ..., -1.3543, -1.3543, -1.3543],
            [ 1.7260,  1.7260,  1.7260,  ..., -1.3689, -1.3689, -1.3689],
            [ 1.7260,  1.7260,  1.7260,  ..., -1.3689, -1.3835, -1.3835]],

           [[ 1.9548,  1.9548,  1.9548,  ...,  2.0449,  2.0449,  2.0449],
            [ 1.9548,  1.9548,  1.9548,  ...,  2.0449,  2.0449,  2.0449],
            [ 1.9548,  1.9548,  1.9548,  ...,  2.0449,  2.0449,  2.0449],
            ...,
            [ 1.8798,  1.8798,  1.8798,  ..., -1.3019, -1.3019, -1.3019],
            [ 1.8798,  1.8798,  1.8798,  ..., -1.3169, -1.3169, -1.3169],
            [ 1.8798,  1.8798,  1.8798,  ..., -1.3169, -1.3319, -1.3319]],

           [[ 2.0037,  2.0037,  2.0037,  ...,  2.1175,  2.1175,  2.1175],


In [ ]:
inputs['aspect_ratio_ids']

tensor([[6]])

In [ ]:
inputs['aspect_ratio_mask']

tensor([[[1, 1, 1, 1]]])

In [ ]:
print(inputs['cross_attention_mask'].shape)
inputs['cross_attention_mask']

torch.Size([1, 17, 1, 4])


tensor([[[[0, 0, 0, 0]],

         [[0, 0, 0, 0]],

         [[0, 0, 0, 0]],

         [[0, 0, 0, 0]],

         [[0, 0, 0, 0]],

         [[1, 1, 1, 1]],

         [[1, 1, 1, 1]],

         [[1, 1, 1, 1]],

         [[1, 1, 1, 1]],

         [[1, 1, 1, 1]],

         [[1, 1, 1, 1]],

         [[1, 1, 1, 1]],

         [[1, 1, 1, 1]],

         [[1, 1, 1, 1]],

         [[1, 1, 1, 1]],

         [[1, 1, 1, 1]],

         [[1, 1, 1, 1]]]])

In [ ]:
model.device

device(type='cpu')

In [ ]:
output = model.generate(**inputs, max_new_tokens=70)
print(processor.decode(output[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
import textwrap

# Decode and format the output
decoded_output = processor.decode(output[0][inputs["input_ids"].shape[-1]:])

# Define the wrap width
wrap_width = 70

# Print formatted output with text wrapping
print("\nFormatted Output:\n")
for line in decoded_output.split("\n"):
    print(textwrap.fill(line, width=wrap_width))

## TSViT Model as MLlamaVisionModel

In [41]:
# import TSViT model modified for integrating to Llama Vision model
from visiontotext.mllamatsvit import MLlamaTSViT

In [42]:
# Loading config file with TSViT parameters
with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)

In [43]:
# Instantiate TSViT modified model
vision_model = MLlamaTSViT(config["MODEL"])
vision_model = vision_model.to(dtype=torch.bfloat16, device='cuda')
vision_model

MLlamaTSViT(
  (to_patch_embedding): Sequential(
    (0): Rearrange('b t c (h p1) (w p2) -> (b h w) t (p1 p2 c)', p1=2, p2=2)
    (1): Linear(in_features=40, out_features=128, bias=True)
  )
  (to_temporal_embedding_input): Linear(in_features=366, out_features=128, bias=True)
  (temporal_transformer): Transformer(
    (layers): ModuleList(
      (0-3): 4 x ModuleList(
        (0): PreNorm(
          (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (fn): Attention(
            (to_qkv): Linear(in_features=128, out_features=384, bias=False)
            (to_out): Sequential(
              (0): Linear(in_features=128, out_features=128, bias=True)
              (1): Dropout(p=0.0, inplace=False)
            )
          )
        )
        (1): PreNorm(
          (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (fn): FeedForward(
            (net): Sequential(
              (0): Linear(in_features=128, out_features=512, bias=True)
              (

In [44]:
# Loading checkpoint on TSViT modified model
vision_model.load_state_dict(torch.load(MODEL_PATH))
vision_model.state_dict()

OrderedDict([('temporal_token',
              tensor([[[-1.4141, -0.7109,  2.9844,  ..., -1.7656, -0.9336, -0.1299],
                       [-0.1289, -0.0417,  3.5781,  ..., -1.2500,  0.1611,  3.5312],
                       [-0.0330, -1.2891, -0.0344,  ..., -0.3359, -2.2969, -1.0547],
                       ...,
                       [ 0.8672, -0.8438, -1.2188,  ..., -1.3594, -1.1484, -1.0469],
                       [ 1.1953, -0.6758, -0.8359,  ...,  1.1797,  0.6406,  1.5625],
                       [ 0.2207,  0.9570,  1.3047,  ...,  3.4219, -2.8125,  1.5312]]],
                     device='cuda:0', dtype=torch.bfloat16)),
             ('space_pos_embedding',
              tensor([[[-0.1934, -0.9648, -0.3340,  ..., -0.8320,  0.2305,  0.4746],
                       [-0.4316, -1.8438, -1.2031,  ..., -0.3691, -3.2656,  1.1406],
                       [-0.1309, -0.8203,  0.6484,  ..., -1.4688, -0.0464,  0.6055],
                       ...,
                       [-1.3125, -1.2891,  0.9

## Llama 3.2 Vision Model with TSViT Model

In [45]:
# Llama Vision Model replaced by TSViT Model
model.model.vision_model = vision_model

In [46]:
# Multi-modal Projector update addapted to TSViT encoder
model.model.multi_modal_projector = nn.Linear(
    config["MODEL"]["num_classes"] * config["MODEL"]["dim"],
    4096,
    bias=True
)

In [47]:
# Multi-modal Projector updated
model.model.multi_modal_projector

Linear(in_features=2432, out_features=4096, bias=True)

In [48]:
model

MllamaForConditionalGeneration(
  (model): MllamaModel(
    (vision_model): MLlamaTSViT(
      (to_patch_embedding): Sequential(
        (0): Rearrange('b t c (h p1) (w p2) -> (b h w) t (p1 p2 c)', p1=2, p2=2)
        (1): Linear(in_features=40, out_features=128, bias=True)
      )
      (to_temporal_embedding_input): Linear(in_features=366, out_features=128, bias=True)
      (temporal_transformer): Transformer(
        (layers): ModuleList(
          (0-3): 4 x ModuleList(
            (0): PreNorm(
              (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (fn): Attention(
                (to_qkv): Linear(in_features=128, out_features=384, bias=False)
                (to_out): Sequential(
                  (0): Linear(in_features=128, out_features=128, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
            )
            (1): PreNorm(
              (norm): LayerNorm((128,), eps=1e-05, elementwise_affi

In [49]:
# Freezing Pre-trained parameters on full model
for param in model.parameters():
    param.requires_grad = False

# Layers on Language model that need to be retrained
cross_attn_layers = [i for i in range(3,39,5)]

# Unfreezing parameters that need been retrained
for i in cross_attn_layers:
    for param in model.model.language_model.layers[i].parameters():
        param.requires_grad = True

# Unfreezing Multi-modal projector
for param in model.model.multi_modal_projector.parameters():
    param.requires_grad = True

In [50]:
model.model.language_model.layers[cross_attn_layers[0]]

MllamaCrossAttentionDecoderLayer(
  (cross_attn): MllamaTextCrossAttention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (q_norm): MllamaTextRMSNorm((128,), eps=1e-05)
    (k_norm): MllamaTextRMSNorm((128,), eps=1e-05)
  )
  (input_layernorm): MllamaTextRMSNorm((4096,), eps=1e-05)
  (mlp): MllamaTextMLP(
    (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
    (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
    (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
    (act_fn): SiLU()
  )
  (post_attention_layernorm): MllamaTextRMSNorm((4096,), eps=1e-05)
)

In [51]:
model.model.language_model.layers[cross_attn_layers[0]].state_dict()

OrderedDict([('cross_attn_attn_gate',
              tensor([0.0005], device='cuda:0', dtype=torch.bfloat16)),
             ('cross_attn_mlp_gate',
              tensor([0.0063], device='cuda:0', dtype=torch.bfloat16)),
             ('cross_attn.q_proj.weight',
              tensor([[-0.0126,  0.0615,  0.0239,  ...,  0.0052, -0.0305,  0.0072],
                      [-0.0258,  0.0115,  0.0017,  ..., -0.0062, -0.0190, -0.0208],
                      [ 0.0034,  0.0068,  0.0231,  ...,  0.0247,  0.0258, -0.0231],
                      ...,
                      [-0.0236,  0.0079, -0.0002,  ...,  0.0337, -0.0016,  0.0087],
                      [ 0.0209,  0.0127,  0.0430,  ..., -0.0155, -0.0613, -0.0286],
                      [-0.0425,  0.0183, -0.0077,  ..., -0.0277, -0.0374, -0.0251]],
                     device='cuda:0', dtype=torch.bfloat16)),
             ('cross_attn.k_proj.weight',
              tensor([[ 0.0090,  0.0253, -0.0376,  ..., -0.0091,  0.0208,  0.0026],
                   

In [52]:
# Calculate Total Quantity of Parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

# Calculate total Quantity of Parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

# Calculate expected quantity of parameters by CrossAttention layer
for i in cross_attn_layers:
    expected_params = sum(
        p.numel()
        for p in model.model.language_model.layers[i].parameters()
        if p.requires_grad
    )
    print(
        f"Trainable parameters for CrossAttention Layer {i}: {expected_params}"
    )

# Calculate expected quantity of parameters for Multi-modal projector layer
expected_params = sum(
    p.numel()
    for p in model.model.multi_modal_projector.parameters()
    if p.requires_grad
)
print(
    f"Trainable parameters for Multi-modal projector Layer : {expected_params}"
)

Total parameters: 9786814352
Trainable parameters: 1754863632
Trainable parameters for CrossAttention Layer 3: 218112258
Trainable parameters for CrossAttention Layer 8: 218112258
Trainable parameters for CrossAttention Layer 13: 218112258
Trainable parameters for CrossAttention Layer 18: 218112258
Trainable parameters for CrossAttention Layer 23: 218112258
Trainable parameters for CrossAttention Layer 28: 218112258
Trainable parameters for CrossAttention Layer 33: 218112258
Trainable parameters for CrossAttention Layer 38: 218112258
Trainable parameters for Multi-modal projector Layer : 9965568


In [ ]:
# Saving Pre-trained Model
model.save_pretrained("datalake/Llama-3.2-11B-TSViT-Instruct")

## Test Interaction with new Model

In [76]:
# Create dummy RGB image of size 24x24
def create_dummy_image(width=24, height=24):
    chn = 3
    data=np.random.randint(low=0,high=256,size=width*height*chn, dtype=np.uint8)
    data=data.reshape(width,height,chn)
    return Image.fromarray(data,'RGB')

image = create_dummy_image()

# Prepare the image input with a text query
messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Can you describe this image?"}
    ]}
]

input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs_model = processor(image, input_text, add_special_tokens=False, return_tensors="pt").to(dtype=torch.bfloat16, device=device)

In [77]:
# changing the 'pixel_values' by the TSViT input sample
inputs_model['pixel_values'] = inputs[sample_idx].unsqueeze(0).to(dtype=torch.bfloat16, device=device)

In [78]:
from torch import autocast

model.to(dtype=torch.bfloat16, device=device)
model.eval()
with torch.no_grad(), autocast(device_type="cuda", dtype=torch.bfloat16):
    output = model.generate(**inputs_model, max_new_tokens=70)
print(processor.decode(output[0][inputs_model["input_ids"].shape[-1]:]))

The image shows a digital rendering of a gray rectangle with rounded corners, set against a gray background. The purpose of the image is to showcase the design and layout of the rectangle.

Here are the key features of the image:

* A gray rectangle with rounded corners:
	+ The rectangle is centered in the image.
	+ It has a light gray
